## Imports

In [70]:
import os
import json
import chromadb
from pathlib import Path
from dotenv import load_dotenv
import re

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from transformers import AutoTokenizer

from anthropic import Anthropic
import gradio as gr
import torch
import time

from neo4j import GraphDatabase
from collections import defaultdict
import base64

import pdfplumber
from docx import Document as DocDoc

import uuid
from pyvis.network import Network

In [71]:
print(gr.__version__)

6.15.2


In [72]:
# Getting environment set up
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Anthropic key loaded: {'Ok' if ANTHROPIC_API_KEY else'Missing'}")
print(f"NEO4J URI: {'Ok' if NEO4J_URI else'Missing'}")
print(f"NEO4J USER: {'Ok' if NEO4J_USERNAME else'Missing'}")
print(f"NEO4J PASSWORD: {'Ok' if NEO4J_PASSWORD else'Missing'}")

CUDA Available: True
Anthropic key loaded: Ok
NEO4J URI: Ok
NEO4J USER: Ok
NEO4J PASSWORD: Ok


## Embedding and Storage for Campaign Notes

In [73]:
CHROMA_PATH = "./chroma_db"
COLLECTION = "TTRPG_corpus"

embedder = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# This corpus store for the static corpus already exists.
corpus_store = Chroma(
    persist_directory=CHROMA_PATH,
    collection_name=COLLECTION,
    embedding_function=embedder
)

# This corpus store is for Campaign notes, and is session only.
campaign_store = Chroma(
    client=chromadb.Client(),
    collection_name="campaign_notes",
    embedding_function=embedder
)

print(f"Corpus chunks loaded: {corpus_store._collection.count()}")
print("Campaign store ready.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Corpus chunks loaded: 15872
Campaign store ready.


## Initial Neo4J Setup

In [74]:
neo4j_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(f"Neo4j driver connected: {'Ok' if neo4j_driver else 'Failed'}")

Neo4j driver connected: Ok


## Client Setup and System Prompt

In [ ]:
client = Anthropic()

system_prompt = '''
You are a creative Game Master assistant. You will be provided with a corpus of novels and TTRPG books to draw from. 
You must primarily use these sources to satisfy the query parameters. 

You may also be given Campaign Notes and Campaign Graph context describing THIS user's own ongoing game. 
Treat established campaign facts (who exists, what has happened, how things relate) as fixed continuity, not something to reinvent or contradict,
but the same as with the corpus, draw on these facts freely as creative inspiration when generating new plot hooks, characters, or scenes. 
If asked directly about the campaign and the answer isn't present in what's provided, say so plainly rather than inventing it.

You may supplement with your own knowledge, but you must consult the corpus for relevant information first. 
Always cite corpus sources.
Corpus sources are cited in the form: title, author [type], or entity [type].
You may deviate from this citation format for direct quotations or in text body citations, but the sources must still be cited.
Cite campaign notes by session and file (e.g., Session 3, session3_notes.txt).
Cite campaign graph entities by name (e.g., Cassius).
Do not offer multiple narrative options. Confidently offer a diversly inspired narrative response.
Avoid creating tables unless needed for stat blocks when asked to produce a character.
'''

## Retrieval

#### Setting up BM25 and Vector Retrieval

In [76]:
# Open corpus_chunks.json again, this is for BM25 search.
with open("corpus_chunks.json", "r", encoding="utf-8") as f:
    unembed_chunks = json.load(f)

corpus_docs = [
    Document(page_content=chunk["text"], metadata=chunk["metadata"])
    for chunk in unembed_chunks
]

# Static Corpus

# Selecting k=8 for BM25 relevant chunks.
bm25_retriever = BM25Retriever.from_documents(corpus_docs, k=8)

# setting up vector retrieval, also for k=8 chunks.
vector_retriever = corpus_store.as_retriever(search_kwargs={"k": 8})

# Combining both retrievers.
corpus_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], weights=[0.5, 0.5]
)

In [77]:
# Campaign Corpus Retrieval 

campaign_docs_session = defaultdict(list)
campaign_retrievers = {}
# Initialization to begin a simple filename hash.
processed_filenames = defaultdict(set)

def build_campaign_retriever(session_id):
    campaign_docs = campaign_docs_session[session_id]
    if not campaign_docs:
        campaign_retrievers.pop(session_id, None)
        return
    bm25_campaign = BM25Retriever.from_documents(campaign_docs, k=5)
    vector_campaign = campaign_store.as_retriever(
        search_kwargs={"k": 5,
                       "filter": {"session_id": session_id}}
    )
    campaign_retrievers[session_id] = EnsembleRetriever(
        retrievers=[bm25_campaign, vector_campaign], weights=[0.5, 0.5]
    )

#### Defining Reciprocol Rank Fusion

In [78]:
def _rrf(ranked_lists, k=60):
    scores = defaultdict(float)
    for rank in ranked_lists:
        for r, doc_id in enumerate(rank, start=1):
            scores[doc_id] += 1.0 / (k + r)

    return sorted(scores, key=lambda x: scores[x], reverse=True)

#### Entity Extraction

In [79]:
# Helper Function to avoid Campaign notes error.
def parse_json_response(raw_text):
    text = raw_text.strip()
    text = re.sub(r"^```\s*(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text.strip())

In [80]:
# Entity Extraction for Neo4j (using Claude Haiku)

# Extracts entities from the query and returns a JSON object.
def extract_entities(text):
    response = client.messages.create(
        model = "claude-haiku-4-5",
        max_tokens = 500,
        messages = [
            {"role": "user",
             "content": f'''Extract all named entities (characters, locations, factions, items, events, monsters, spells, etc.), 
             from the text. Only return a JSON array of the entity names, nothing additional.
             Text: {text}'''
            }
        ]
    )
    return parse_json_response(response.content[0].text)

#### Graph Retrieval

In [ ]:
# Graph Retrieve

# Takes in the query, but also the session_id to partition if two people use it at the same time.
def graph_retrieve(query, session_id, campaign_results=None, max_hops=3):
    if not campaign_docs_session[session_id]:
        return []
    entities = extract_entities(query)
    if not entities and campaign_results:
        seed_text = " ".join(doc.page_content for doc in campaign_results[:3])
        entities = extract_entities(seed_text)
    if not entities:
        return []
    
    with neo4j_driver.session() as neo4j_session:
        # Limits to 10 entites and 10 paths.
        result = neo4j_session.run(
            f'''
            MATCH (e:Entity {{session_id: $session_id}})
            WHERE any(name IN $entities WHERE tolower(e.name) CONTAINS toLower(name))
            OPTIONAL MATCH path = (e)-[rels*1..{max_hops}]-(neighbor:Entity {{session_id: $session_id}})
            WHERE neighbor IS NULL OR neighbor <> e
            WITH e, collect(DISTINCT CASE WHEN path IS NULL THEN NULL ELSE {{
                   nodes: [n IN nodes(path) | n.name],
                   rels: [r IN relationships(path) | r.relation]
                   }} END) AS all_paths
            RETURN e.name AS name, e.type AS type, e.description AS description,
                   all_paths[0..10] AS paths
            ORDER BY size(all_paths) DESC
            LIMIT 10
            ''',
            entities=entities,
            session_id=session_id
        )
        docs = []
        for record in result:
            content = f"{record['name']} ({record['type']})\n{record['description']}"
            chains = []
            for p in record['paths']:
                if not p or not p.get('nodes'):
                    continue
                nodes, rels = p['nodes'], p['rels']
                chain = nodes[0]
                for rel, nxt in zip(rels, nodes[1:]):
                    chain += f" --[{rel}]--> {nxt}"
                if len(nodes) > 1:
                    chains.append(chain)
            if chains:
                content += "\nConnections:\n" + "\n".join(f"- {c}" for c in chains)
            docs.append(Document(
                page_content=content, 
                metadata={"source": record['name'],
                          "type": "campaign_graph",
                          "session_id": session_id}
            ))
    return docs

In [82]:
# For clearing the Neo4j and campaign notes sessions:
def clear_session(session_id):
    if not isinstance(session_id, str):
        return
    with neo4j_driver.session() as neo4j_session:
        neo4j_session.run(
            "MATCH (e:Entity {session_id: $session_id}) DETACH DELETE e",
            session_id=session_id
        )
    # For clearing Campaign stores:
    results = campaign_store.get(where = {"session_id": session_id})
    if results["ids"]:
        campaign_store.delete(ids=results["ids"])
    campaign_docs_session.pop(session_id, None)
    campaign_retrievers.pop(session_id, None)
    processed_filenames.pop(session_id, None)

#### Full Retrieval with RRF

In [83]:
# Sometimes, a single source dominates the entire retrieval process. This only allows 3 chunksper source.
def deduplicate_by_source(docs, max_per_source=3):
    seen = {}
    result = []
    for doc in docs:
        source = doc.metadata.get("source")
        seen[source] = seen.get(source, 0)
        if seen[source] < max_per_source:
            result.append(doc)
            seen[source] += 1
    return result

In [ ]:
def retrieve(query, session_id):
    corpus_results = deduplicate_by_source(corpus_retriever.invoke(query))

    campaign_ret = campaign_retrievers.get(session_id)
    campaign_results = campaign_ret.invoke(query) if campaign_ret else []

    graph_results = graph_retrieve(query, session_id, campaign_results=campaign_results)

    if not campaign_results and not graph_results:
        return corpus_results
    
    doc_map = {}
    all_lists = []

    for results in [campaign_results, graph_results, corpus_results]:
        if results:
            ids = [id(doc) for doc in results]
            for i, doc in zip(ids, results):
                doc_map[i] = doc
            all_lists.append(ids)

    merged_res = _rrf(all_lists)
    return [doc_map[i] for i in merged_res if i in doc_map]

### Campaign Ingestion

In [85]:
# Make a utility function to handle may file types for text extraction.
def extract_text(file):
    path = file.name
    # Find what the file extension is.
    ext = Path(path).suffix.lower()

    if ext in (".txt", ".md"):
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
        
    elif ext == ".pdf":
        text = ""
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text
        return text

    elif ext in (".docx", ".doc"):
        doc = DocDoc(path)
        return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

    else:
        return ""

In [86]:
def extract_campaign_meta(txt):
    response = client.messages.create(
        model = "claude-haiku-4-5",
        max_tokens = 8000,
        messages = [
            {"role": "user",
             "content": f'''
             Analyze the TTRPG campaign notes and return only a JSON object containing:
             - "tags": list of named entities (so, characters, locations, organizations/factions, etc.)
             - "entities": list of objects with "name", "type", and "description".
             - "relationships": list of objects with "source", "target", and "relation".
             Text: {txt}'''
            }
        ]
    )
    if response.stop_reason == "max_tokens":
        raise ValueError("Max tokens exceeded. Add tokens or put in smaller chunks of text")
    return parse_json_response(response.content[0].text)

In [87]:
def writing_graph_entities(entities, relationships, session_id):
    with neo4j_driver.session() as neo4j_session:
        for entity in entities:
            neo4j_session.run('''
            MERGE (e:Entity {name: $name, session_id: $session_id})
            SET e.type = $type, e.description = $description
            ''',
            name=entity['name'],
            session_id=session_id,
            type=entity.get("type", "unknown"),
            description=entity.get("description", ""),
            )
        for rel in relationships:
            neo4j_session.run('''
            MATCH (a:Entity {name: $source, session_id: $session_id})
            MATCH (b:Entity {name: $target, session_id: $session_id})
            MERGE (a)-[r:RELATED_TO {session_id: $session_id}]->(b)
            SET r.relation = $relation
            ''',
            source=rel['source'],
            target=rel['target'],
            session_id=session_id,
            relation=rel.get("relation", "related_to")
            )

In [88]:
# Make splitter for the campaign notes. Smaller than the corpus, because it is likely more factually dense.
campaign_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
campaign_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    campaign_tokenizer,
    chunk_size = 250,
    chunk_overlap = 25
    )

In [ ]:
def ingest_campaign_notes(files, session_number, session_id):
    # Campaign notes are optional, so may be none sometimes.
    if not files:
        return "No campaign notes provided."
    
    # Doing a simple hash. Just on file name.
    processed, skipped = 0, 0
    # Making an index and getting files.
    for file_idx, file in enumerate(files):
        filename = Path(file.name).name
        if filename in processed_filenames[session_id]:
            skipped += 1
            continue
        processed_filenames[session_id].add(filename)

        text = extract_text(file)
        if not text.strip():
            continue

        # using the metadata extractor and writing to neo4j
        meta = extract_campaign_meta(text)
        writing_graph_entities(meta["entities"], meta["relationships"], session_id)

        chunks = campaign_splitter.create_documents([text])
        total = len(chunks)

        docs = []
        for i, chunk in enumerate(chunks):
            chunk.metadata.update({
                "session_id": session_id,
                "session": int(session_number) if session_number is not None else file_idx,
                "source": filename,
                "chunk_index": i,
                "temporal_position": i / total,
                "tags": ", ".join(meta["tags"])
            })
            docs.append(chunk)
        
        campaign_store.add_documents(docs)
        campaign_docs_session[session_id].extend(docs)
        # Processed for hash.
        processed += 1
    
    build_campaign_retriever(session_id)
    msg = f"Ingested {processed} campaign note(s)."
    if skipped:
        msg += f" Skipped {skipped} file(s) with a name already used this session."
    return msg

## Inference

In [ ]:
# Putting the context in a string format prompt so it can be passed to the LLM.
def format_context(docs):
    chunks = []
    for doc in docs:
        meta = doc.metadata
        if meta.get("type") == "campaign_graph":
            source_label = f"[Campaign Graph: {meta.get('source', 'Unknown')}]"
        elif "session_id" in meta:
            source_label = f"[Campaign Notes: Session {meta.get('session', 'Unknown')} -- {meta.get('source', 'Unknown')}]"
        else:
            source_label = f"[{meta.get('title', 'Unknown')}] by {meta.get('author', 'Unknown')} [{meta.get('type', 'Unknown')}]"
        chunks.append(f"{source_label}\n{doc.page_content}")
    return "\n\n".join(chunks)

def generate(query, history, session_id):
    # Call the merged retriever and call format_context to format the context.
    docs = retrieve(query, session_id)
    context = format_context(docs)

    # Format the history for the LLM and assemble message.
    messages = [
        *[{"role": m["role"], "content": m["content"]} for m in history],
        {"role": "user", 
         "content": f"Context:\n{context}\n\nQuery: {query}"}
    ]

    # Call the LLM
    response = client.messages.create(
        model = "claude-sonnet-4-6",
        max_tokens = 2000,
        system = system_prompt,
        messages = messages
    )

    # Checking for how many output tokens are used.
    print(f"Output Tokens: {response.usage.output_tokens}")
    return response.content[0].text

## User Interface

In [91]:
# UI Helper Functions

# Generating the Neo4j Graph for the campaign.
# Changing graph size.
GRAPH_HEIGHT_PX = 700 

# Matching to color nodes.
def node_color(entity_type):
    if not entity_type:
        return "#6fa8dc"  
    t = entity_type.lower()
    if any(k in t for k in ("creature", "monster")):
        return "#e05252"  
    if any(k in t for k in ("character", "person", "npc")):
        return "#f4d35e"  
    if any(k in t for k in ("location", "place", "region", "city", "town", "building")):
        return "#b185db"  
    return "#6fa8dc"  

def build_graph_HTML(session_id):
    with neo4j_driver.session() as neo4j_session:
        result = neo4j_session.run('''
        MATCH (e:Entity {session_id: $session_id})
        OPTIONAL MATCH (e)-[r:RELATED_TO]->(neighbor:Entity {session_id: $session_id})
        RETURN e.name AS source, e.type AS source_type,
                                   neighbor.name AS target, neighbor.type AS target_type,
                                   r.relation AS relation''',
                                   session_id=session_id)
        records = list(result)
    
    if not records:
        return "<p style='color:gray'>No campaign entities yet.</p>"
    
    net = Network(height=f"{GRAPH_HEIGHT_PX}px", width="100%", bgcolor="#1a1a2e", font_color="white")
    net.barnes_hut(gravity=-15000, central_gravity=0.25, spring_length=220,
                    spring_strength=0.015, damping=0.12)

    seen = set()
    for record in records:
        if record["source"] not in seen:
            net.add_node(record["source"], label=record["source"],
                         title=record["source_type"] or "",
                         color=node_color(record["source_type"]))
            seen.add(record["source"])
        if record["target"] and record["target"] not in seen:
            net.add_node(record["target"], label=record["target"],
                         title=record["target_type"] or "",
                         color=node_color(record["target_type"]))
            seen.add(record["target"])
        if record["target"]:
            net.add_edge(record["source"], record["target"], label=record["relation"] or "")

    legend = (
        '<div style="font-family:sans-serif;font-size:12px;padding:6px 10px;'
        'background:#12121f;border-radius:6px;margin-bottom:6px;'
        'display:flex;gap:16px;align-items:center">'
        '<span style="color:#e0e0e0"><span style="display:inline-block;width:12px;height:12px;border-radius:50%;'
        'background:#f4d35e;vertical-align:middle;margin-right:4px"></span>People</span>'
        '<span style="color:#e0e0e0"><span style="display:inline-block;width:12px;height:12px;border-radius:50%;'
        'background:#e05252;vertical-align:middle;margin-right:4px"></span>Creatures</span>'
        '<span style="color:#e0e0e0"><span style="display:inline-block;width:12px;height:12px;border-radius:50%;'
        'background:#b185db;vertical-align:middle;margin-right:4px"></span>Places</span>'
        '<span style="color:#e0e0e0"><span style="display:inline-block;width:12px;height:12px;border-radius:50%;'
        'background:#6fa8dc;vertical-align:middle;margin-right:4px"></span>Other</span>'
        '</div>'
    )
    
    html_out = net.generate_html()
    encoded = base64.b64encode(html_out.encode()).decode()
    return (
        legend +
        f'<iframe src="data:text/html;base64,{encoded}" '
        f'width="100%" height="{GRAPH_HEIGHT_PX}px" frameborder="0" '
        f'style="border-radius:8px;border:1px solid #444"></iframe>'
    )

def chat(message, history, request: gr.Request):
    print("session_hash in chat():", request.session_hash)
    session_id = request.session_hash
    start = time.time()
    response = generate(message, history, session_id)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})
    elapsed = time.time() - start
    return history, "", f"{elapsed:.2f}s"

def tick(start):
    if start is None:
        return "0.00s"
    return f"{time.time() - start:.2f}s"

def ingest(files, session_num, request: gr.Request):
    session_id = request.session_hash
    status = ingest_campaign_notes(files, session_num, session_id)
    graph_html = build_graph_HTML(session_id)
    return status, graph_html, None

In [ ]:
theme_toggle_js = """
() => {
    document.body.classList.toggle('dark');
}
"""

custom_css = """
.gradio-container {
    background: url('/gradio_api/file=assets/adventure_day.png');
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
}
.block {
    background: rgba(255, 255, 255, 0.85) !important;
}
body.dark .gradio-container {
    background: url('/gradio_api/file=assets/adventure_night.png');
    background-size: cover;
    background-position: center;
}
body.dark .block {
    background: rgba(20, 20, 30, 0.75) !important;
}
.header-text, .header-text * {
    color: #2b1d0e !important;
}
body.dark .header-text, body.dark .header-text * {
    color: #f2e8d5 !important;
}
"""

def cleanup_on_unload(request: gr.Request):
    clear_session(request.session_hash)
    print(f"Session {request.session_hash} cleaned up on tab close.")

with gr.Blocks(title="Petrichor") as demo:
    with gr.Row():
        gr.Markdown('''
            # Petrichor
            ### *The smell of rain before your storm*
            *A TTRPG narrative helper*
        ''', elem_classes = "header-text")
        theme_btn = gr.Button("🌓 Toggle theme", scale=0, size="sm")

    theme_btn.click(None, [], [], js=theme_toggle_js)

    with gr.Tab("Chat"):
        chatbot = gr.Chatbot(height=500)
        with gr.Row():
            msg_box = gr.Textbox(
                placeholder="Ask your TTRPG creative assistant...",
                show_label=False,
                scale=5
            )
            submit_btn = gr.Button("Ask", variant="primary", scale=1)
            timer_display = gr.Textbox(value = "0.00s", interactive=False, scale=1, show_label=False)
            tick_timer = gr.Timer(0.1, active=False)
            timer_start = gr.State(None)
        clear_btn = gr.Button("Clear Chat")

    with gr.Tab("Campaign Notes"):
        files = gr.File(file_count="multiple", label="Upload campaign notes")
        session_num = gr.Number(label="Session Number (Optional)", precision=0)
        process_btn  = gr.Button("Process Campaign Notes", variant="primary")
        clear_session_btn = gr.Button("Clear Session",variant="stop")
        processing_status = gr.Textbox(label="Status", interactive=False)

    with gr.Tab("Campaign Graph"):
        graph_out = gr.HTML()

    for trigger in [submit_btn.click, msg_box.submit]:
        trigger(
            fn=lambda: (time.time(), gr.Timer(active=True)),
            outputs=[timer_start, tick_timer]
        ).then(
            fn=chat,
            inputs=[msg_box, chatbot],
            outputs=[chatbot, msg_box, timer_display]
        ).then(
            fn=lambda: gr.Timer(active=False),
            outputs=[tick_timer]
        )
    tick_timer.tick(fn=tick, inputs=[timer_start], outputs=[timer_display])
    clear_btn.click(fn=lambda: [], outputs=[chatbot])
    process_btn.click(
        fn=ingest,
        inputs=[files, session_num],
        outputs=[processing_status, graph_out, files]
    )

    def _handle_clear_session(request: gr.Request):
        clear_session(request.session_hash)
        return "Session cleared.", "<p style='color:gray'>No campaign entities yet.</p>", []

    clear_session_btn.click(
        fn=_handle_clear_session,
        inputs=[],
        outputs=[processing_status, graph_out, chatbot]
    )

    demo.unload(cleanup_on_unload)

demo.launch(theme="harsh8001/comic", inbrowser=True, css=custom_css, allowed_paths=["assets"])

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Session ra0jo2s7hj cleaned up on tab close.
Session pyul1o2q6oa cleaned up on tab close.
Session setqdnitcgn cleaned up on tab close.


## Testing